# Ingest → Embed Pipeline Integration Test

Verifies the end-to-end pipeline:
1. Fetch IPF papers from PubMed via `search_pubmed`
2. Embed and store in ChromaDB via `embed_and_store`
3. Query the collection and inspect cosine distances

**Pass criteria:** 25 papers fetched, 25 upserted, query distances < 0.15 for top hits.

In [1]:
import sys, pathlib
# Kernel cwd is the notebook's directory (test/); parent is the project root
_root = pathlib.Path.cwd().parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print("Project root:", _root)


Project root: /Users/richardahn/projects/fibrosisLit


In [2]:
import logging, pandas as pd
from pipeline.ingest import search_pubmed
from pipeline.embed import embed_and_store, query
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s — %(message)s")

In [3]:
papers = search_pubmed(
    "idiopathic pulmonary fibrosis[MeSH] TGF-beta myofibroblast",
    max_results=25
)
print(f"Fetched {len(papers)} papers")

INFO pipeline.ingest — Searching PubMed: 'idiopathic pulmonary fibrosis[MeSH] TGF-beta myofibroblast' (max 25)


INFO pipeline.ingest — ESearch returned 25 PMIDs


INFO pipeline.ingest — Fetching 25 PMIDs from NCBI


INFO pipeline.ingest — Fetched 25 papers


Fetched 25 papers


In [4]:
n = embed_and_store(papers)
print(f"Upserted {n} documents into ChromaDB")

INFO pipeline.embed — Initializing ChromaDB at ./chroma_db


INFO chromadb.telemetry.product.posthog — Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.


INFO pipeline.embed — ChromaDB collection 'fibrosis_papers' ready (25 documents)


INFO pipeline.embed — Embedding 25 papers with SPECTER2 (proximity)


INFO pipeline.embed — Loading SPECTER2 tokenizer and base model


...

INFO adapters.loading — No matching prediction head found in '/Users/richardahn/.cache/huggingface/hub/models--allenai--specter2/snapshots/2081559630a80fc5851d8f798a05ba81e9468089'


INFO adapters.heads.model_mixin — Could not identify valid prediction head(s) from setup 'Stack[[PRX]]'.


INFO pipeline.embed — SPECTER2 model ready


INFO pipeline.embed — Upserted 25 documents into ChromaDB collection 'fibrosis_papers'


Upserted 25 documents into ChromaDB


In [5]:
results = query("SPP1 macrophage myofibroblast activation IPF", n_results=25)

In [6]:
rows = []
for rank, result in enumerate(results, start=1):
    rows.append({
        "Rank": rank,
        "PMID": result.get("pmid", ""),
        "Title": result.get("title", ""),
        "Journal": result.get("journal", ""),
        "Year": result.get("pub_date", "")[:4] if result.get("pub_date") else "",
        "Distance": round(result["distance"], 4),
    })

df = pd.DataFrame(rows).sort_values("Distance").reset_index(drop=True)
df["Rank"] = df.index + 1

pd.set_option("display.max_colwidth", 80)
display(df)


,Rank,PMID,Title,Journal,Year,Distance
0,1,40222273,Idiopathic pulmonary fibrosis microenvironment: Novel mechanisms and researc...,International immunopharmacology,2025,0.0593
1,2,38212077,Sfrp1 inhibits lung fibroblast invasion during transition to injury-induced ...,The European respiratory journal,2024,0.0610
2,3,37707699,"Idiopathic pulmonary fibrosis (IPF): disease pathophysiology, targets, and p...",Molecular and cellular biochemistry,2024,0.0659
3,4,29873047,Idiopathic pulmonary fibrosis:Pathophysiological data.,La Tunisie medicale,2017,0.0752
4,5,39026235,Regulation of myofibroblast dedifferentiation in pulmonary fibrosis.,Respiratory research,2024,0.0760
5,6,39319990,"MEOX1 triggers myofibroblast apoptosis resistance, contributing to pulmonary...",Journal of cellular physiology,2024,0.0784
6,7,35427207,ATF3 -activated accelerating effect of LINC00941/lncIAPF on fibroblast-to-my...,Autophagy,2022,0.0790
7,8,38122910,The mechanism of Shenlong Jianji treatment of idiopathic pulmonary fibrosis ...,Journal of ethnopharmacology,2024,0.0798
8,9,37653024,Autocrine TGF-β-positive feedback in profibrotic AT2-lineage cells plays a c...,Nature communications,2023,0.0813
9,10,40675771,Integrated spatial and single-cell transcriptomics reveal PAK kinase as a th...,The European respiratory journal,2025,0.0823


## Summary

**What a passing run looks like:**
- Cell 3: `Fetched 25 papers`
- Cell 4: `Upserted 25 documents into ChromaDB`
- Cell 6 table: all 25 papers visible, top hits have `Distance < 0.15`

High distances (> 0.4) for the bottom rows are expected — those papers are less
semantically similar to the SPP1/macrophage query but were still ingested correctly.
The important check is that *all 25 rows appear* and distances are finite.